## Machine Learning Project 1

Predict heart disease with regularized logistic regression. Select preprocessing, penalties, blend weights, and the threshold using the training set.


In [1]:
import gc
import os
from itertools import combinations, product
from pathlib import Path

os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "4")

import numpy as np

from helpers import create_csv_submission
from logistic_model import (
    SurveyFeatures,
    best_f1_threshold,
    blend_logistic_probabilities,
    fit_elastic_logistic,
    fit_logistic,
    predict_probabilities,
    stratified_folds,
    stratified_split,
)

DATA_DIR = Path("data/dataset")


### Load the dataset

Extract the dataset into `data/dataset`. Cache each feature CSV separately to speed up later runs.


In [2]:
def load_features(filename, force_reload=False):
    """Read a feature CSV, keeping its IDs and column names separate."""
    cache_file = filename.with_suffix(".npz")
    if (
        not force_reload
        and cache_file.is_file()
        and cache_file.stat().st_mtime >= filename.stat().st_mtime
    ):
        with np.load(cache_file, allow_pickle=False) as cached:
            return cached["x"], cached["ids"], cached["columns"].tolist()

    with filename.open() as file:
        columns = file.readline().strip().split(",")[1:]
    data = np.genfromtxt(filename, delimiter=",", skip_header=1)
    ids, x = data[:, 0].astype(np.int64), data[:, 1:]
    np.savez_compressed(cache_file, x=x, ids=ids, columns=columns)
    return x, ids, columns


### Validation split

Use a stratified 80/20 split. Fit preprocessing on the fitting rows of each split.


In [3]:
def validation_split(y, val_ratio=0.2, seed=42):
    """Return fitting and validation row indices, preserving class proportions."""
    return stratified_split(y, seed=seed, validation_ratio=val_ratio)


### Preprocess the data

Compare categorical health-day responses, rank scaling, and focused splines with the original encodings. Also test decoding health-day values according to the [CDC codebook](https://www.cdc.gov/brfss/annual_data/2015/pdf/codebook15_llcp.pdf). Learn all statistics from the fitting rows.


In [4]:
def preprocess_x(x_train, columns, features):
    """Learn an encoding on training rows and return it with the encoded data."""
    encoder = SurveyFeatures(**features)
    return encoder.fit_transform(x_train, columns), encoder


def preprocess_y(y):
    return (y == 1).astype(np.float32)


### Train on the data

Compare L2 and elastic-net logistic regression. Initialize elastic net with an L2 fit; leave the intercept unpenalized.


In [5]:
def train(
    x_train,
    y_train,
    params,
    max_iters=800,
    initial_w=None,
    verbose=False,
):
    """Fit one model using the selected parameters."""
    if params["l1"] == 0:
        return fit_logistic(
            x_train,
            y_train,
            lambda_=params["l2"],
            max_iters=max_iters,
            initial_w=initial_w,
            verbose=verbose,
        )
    if initial_w is None:
        initial_w, _ = fit_logistic(
            x_train, y_train, lambda_=1e-3, max_iters=400, verbose=verbose
        )
    return fit_elastic_logistic(
        x_train,
        y_train,
        l1=params["l1"],
        lambda_=params["l2"],
        max_iters=max_iters,
        initial_w=initial_w,
        verbose=verbose,
    )


### Predict on data


In [6]:
def blend_probabilities(probabilities, weights, method="logit"):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if method == "probability":
        return np.average(probabilities, axis=0, weights=weights)
    return blend_logistic_probabilities(probabilities, weights)


def predict(probabilities, threshold=0.5):
    """Return submission labels in {-1, 1}."""
    return np.where(probabilities >= threshold, 1, -1)


### Evaluate


In [7]:
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)


def f1_score(y_true, y_pred):
    true_positive = np.sum((y_true == 1) & (y_pred == 1))
    false_positive = np.sum((y_true != 1) & (y_pred == 1))
    false_negative = np.sum((y_true == 1) & (y_pred != 1))
    denominator = 2 * true_positive + false_positive + false_negative
    return 2 * true_positive / denominator if denominator > 0 else 0.0


In [8]:
def cross_validate(x, y, columns, features, params, folds, max_iters=800):
    """Predict each training row with a model fitted on the other folds."""
    probabilities = np.empty(len(y), dtype=np.float32)
    for fold, val_idx in enumerate(folds, start=1):
        fit_mask = np.ones(len(y), dtype=bool)
        fit_mask[val_idx] = False
        x_fit, encoder = preprocess_x(x[fit_mask], columns, features)
        w, info = train(x_fit, y[fit_mask], params, max_iters)
        del x_fit
        x_val = encoder.transform(x[val_idx])
        probabilities[val_idx] = predict_probabilities(x_val, w)
        print(
            f"Fold {fold}/{len(folds)}: {info['status']}, {info['iterations']} iterations",
            flush=True,
        )
        del x_val, w, encoder
        gc.collect()
    return probabilities


### Explore

Load the training data and create the validation split.


In [9]:
x_train_raw, ids_train, columns = load_features(DATA_DIR / "x_train.csv")
training_labels = np.loadtxt(
    DATA_DIR / "y_train.csv", delimiter=",", skiprows=1, dtype=int
)
if not np.array_equal(training_labels[:, 0], ids_train):
    raise ValueError("Training features and labels are not aligned.")
y_train_raw = training_labels[:, 1]
print(
    f"Training: {x_train_raw.shape}; positive labels: {(y_train_raw == 1).mean():.2%}"
)


Training: (328135, 321); positive labels: 8.83%


In [10]:
y_train = preprocess_y(y_train_raw)
train_idx, val_idx = validation_split(y_train, val_ratio=0.2, seed=42)
y_fit, y_val = y_train[train_idx], y_train[val_idx]
print(f"Fitting rows: {len(train_idx):,}; validation rows: {len(val_idx):,}")


Fitting rows: 262,508; validation rows: 65,627


First compare preprocessing with the same reference penalties. Keep the linear and binned baselines plus the two best alternatives, then search their L1 and L2 penalties. A zero L1 penalty gives L2 logistic regression.


In [11]:
health_days = ("PHYSHLTH", "MENTHLTH", "POORHLTH")
feature_options = {
    "linear": dict(mode="linear", interactions=True),
    "bins": dict(mode="bins", interactions=True),
    "days": dict(mode="linear", interactions=True, categorical_columns=health_days),
    "rank": dict(mode="rank", interactions=True, categorical_columns=health_days),
    "clinical_splines": dict(
        mode="spline",
        interactions=True,
        categorical_columns=health_days,
        nonlinear_columns=("_AGE80", "_BMI5"),
        n_knots=5,
        nonlinear_interactions=True,
    ),
}
feature_options.update(
    clean_days=dict(mode="linear", interactions=True, clean_health_days=True),
    clean_splines=dict(
        mode="spline",
        interactions=True,
        clean_health_days=True,
        nonlinear_columns=("_AGE80", "_BMI5", *health_days),
        n_knots=5,
        nonlinear_interactions=True,
    ),
)
reference_params = dict(l1=3e-4, l2=1e-4)
l1_values = (0.0, 1e-4, 3e-4, 1e-3)
l2_values = (1e-4, 1e-3)
max_iters = 800
n_folds = 3
cv_seed = 2026
blend_methods = ("probability", "logit")


In [ ]:
preprocessing_results = []
for name, features in feature_options.items():
    x_fit, encoder = preprocess_x(x_train_raw[train_idx], columns, features)
    n_features = x_fit.shape[1]
    w, info = train(x_fit, y_fit, reference_params, max_iters)
    del x_fit
    x_val = encoder.transform(x_train_raw[val_idx])
    candidate_threshold, score = best_f1_threshold(
        y_val, predict_probabilities(x_val, w)
    )
    preprocessing_results.append(
        dict(name=name, f1=score, threshold=candidate_threshold)
    )
    print(f"{name}: validation F1={score:.6f}, {n_features} features", flush=True)
    del x_val, w, encoder
    gc.collect()

alternatives = [
    row for row in preprocessing_results if row["name"] not in ("linear", "bins")
]
shortlisted_models = ("linear", "bins") + tuple(
    row["name"]
    for row in sorted(alternatives, key=lambda row: row["f1"], reverse=True)[:2]
)
print("Selected encodings:", shortlisted_models)


linear: validation F1=0.449939, 2222 features


In [ ]:
search_results = []
best_params = {}
for name in shortlisted_models:
    x_fit, encoder = preprocess_x(
        x_train_raw[train_idx], columns, feature_options[name]
    )
    x_val = encoder.transform(x_train_raw[val_idx])
    initial_w, _ = fit_logistic(x_fit, y_fit, lambda_=1e-3, max_iters=400)
    results = []
    for l2, l1 in product(l2_values, l1_values):
        params = dict(l1=l1, l2=l2)
        w, info = train(x_fit, y_fit, params, max_iters, initial_w=initial_w)
        candidate_threshold, score = best_f1_threshold(
            y_val, predict_probabilities(x_val, w)
        )
        results.append(dict(l1=l1, l2=l2, threshold=candidate_threshold, f1=score))
        print(f"{name}: L1={l1:g}, L2={l2:g}, F1={score:.6f}", flush=True)
    best_params[name] = max(results, key=lambda row: row["f1"])
    search_results.extend(dict(name=name, **row) for row in results)
    del x_fit, x_val, w, encoder
    gc.collect()


In [ ]:
for name, params in best_params.items():
    print(
        f"{name}: L1={params['l1']:g}, L2={params['l2']:g}, validation F1={params['f1']:.6f}"
    )


Use three-fold predictions to choose the blend and threshold. These are tuning scores because they guide model selection.


In [ ]:
folds = stratified_folds(y_train, n_folds=n_folds, seed=cv_seed)
oof_probabilities = {}
for name in shortlisted_models:
    print(f"Cross-validating {name}", flush=True)
    oof_probabilities[name] = cross_validate(
        x_train_raw,
        y_train,
        columns,
        feature_options[name],
        best_params[name],
        folds,
        max_iters,
    )


In [ ]:
# Compare 10% weight steps across all models, then refine pairs in 5% steps.
weight_grid = []
for partial in product(range(11), repeat=len(shortlisted_models) - 1):
    if sum(partial) <= 10:
        weight_grid.append(np.array((*partial, 10 - sum(partial))) / 10)
for i, j in combinations(range(len(shortlisted_models)), 2):
    for weight in np.linspace(0, 1, 21):
        weights = np.zeros(len(shortlisted_models))
        weights[i], weights[j] = weight, 1 - weight
        weight_grid.append(weights)

components = [oof_probabilities[name] for name in shortlisted_models]
blend_results = []
for method, candidate_weights in product(blend_methods, weight_grid):
    probabilities = blend_probabilities(components, candidate_weights, method)
    candidate_threshold, score = best_f1_threshold(y_train, probabilities)
    blend_results.append(
        dict(
            method=method,
            weights=tuple(candidate_weights),
            threshold=candidate_threshold,
            f1=score,
        )
    )

best_blend = max(blend_results, key=lambda row: row["f1"])
model_names = shortlisted_models
selected_weights = dict(zip(model_names, best_blend["weights"]))
threshold = best_blend["threshold"]


In [ ]:
print(f"Blend method: {best_blend['method']}")
for name, weight in selected_weights.items():
    if weight > 0:
        print(f"{name}: {weight:.0%}")
print(f"Threshold: {threshold:.9f}")
print(f"Out-of-fold tuning F1: {best_blend['f1']:.6f}")


### Final training

Refit the selected models on all training rows. Keep the blend weights and threshold fixed.


In [14]:
x_test_raw, ids_test, test_columns = load_features(DATA_DIR / "x_test.csv")
if columns != test_columns:
    raise ValueError("Training and test feature columns differ.")


In [15]:
models = {}
test_probabilities = {}
for name in model_names:
    if selected_weights[name] == 0:
        continue
    print(f"Fitting {name} on {len(y_train):,} rows", flush=True)
    x_fit, encoder = preprocess_x(x_train_raw, columns, feature_options[name])
    models[name], info = train(
        x_fit, y_train, best_params[name], max_iters, verbose=True
    )
    del x_fit
    x_test = encoder.transform(x_test_raw)
    test_probabilities[name] = predict_probabilities(x_test, models[name])
    print(
        f"Final {name}: {info['status']}, {info['iterations']} iterations", flush=True
    )
    del x_test, encoder
    gc.collect()


Fitting linear on 328,135 rows
  Iteration 50: objective=0.212755
  Iteration 100: objective=0.212644
  Iteration 100: objective=0.216193, nonzero=942
  Iteration 200: objective=0.215988, nonzero=645
  Iteration 300: objective=0.215940, nonzero=611
  Iteration 400: objective=0.215929, nonzero=595
  Iteration 500: objective=0.215925, nonzero=595
  Iteration 600: objective=0.215923, nonzero=595
  Iteration 700: objective=0.215923, nonzero=595
Final linear: converged, 712 iterations


In [16]:
probabilities = blend_probabilities(
    list(test_probabilities.values()),
    [selected_weights[name] for name in test_probabilities],
    best_blend["method"],
)
test_predictions = predict(probabilities, threshold)


### Submission


In [17]:
create_csv_submission(ids_test, test_predictions, "submission.csv")
print(f"Saved submission.csv: {len(test_predictions):,} rows")
print(f"Positive predictions: {(test_predictions == 1).mean():.2%}")


Saved submission.csv: 109,379 rows
Positive predictions: 12.56%
